# Anomaly Detection using PyOD

## Assignment (e): Demonstrate Anomaly Detection using PyOD

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. Introduction
2. PyOD Overview
3. Univariate Anomaly Detection
4. Multivariate Anomaly Detection
5. Time Series Anomaly Detection
6. Credit Card Fraud Detection Use Case
7. Model Comparison and Evaluation
8. Conclusion

In [ ]:
!pip install pyod numpy pandas matplotlib seaborn scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.utils.data import generate_data
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Libraries imported successfully!")

## 1. Generate Synthetic Data with Outliers

In [ ]:
contamination = 0.1
X_train, X_test, y_train, y_test = generate_data(n_train=500, n_test=200, n_features=2, contamination=contamination, random_state=42)
print(f"Training: {X_train.shape}, Test: {X_test.shape}")
print(f"Outliers - Train: {y_train.sum()}, Test: {y_test.sum()}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='blue', alpha=0.6, label='Normal')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='red', marker='x', s=100, label='Outlier')
plt.title('Training Data'); plt.legend()
plt.subplot(1, 2, 2)
plt.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], c='blue', alpha=0.6, label='Normal')
plt.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], c='red', marker='x', s=100, label='Outlier')
plt.title('Test Data'); plt.legend()
plt.tight_layout(); plt.show()

## 2. Univariate Anomaly Detection

In [ ]:
normal_data = np.random.normal(50, 10, 900)
anomalies = np.concatenate([np.random.normal(10, 3, 50), np.random.normal(90, 3, 50)])
univariate_data = np.concatenate([normal_data, anomalies])
univariate_labels = np.concatenate([np.zeros(900), np.ones(100)])
shuffle_idx = np.random.permutation(len(univariate_data))
univariate_data, univariate_labels = univariate_data[shuffle_idx], univariate_labels[shuffle_idx]
X_uni = univariate_data.reshape(-1, 1)

iforest_uni = IForest(contamination=0.1, random_state=42)
iforest_uni.fit(X_uni)
y_pred_uni = iforest_uni.labels_
print(f"Univariate ROC-AUC: {roc_auc_score(univariate_labels, iforest_uni.decision_scores_):.4f}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
colors = ['red' if l == 1 else 'blue' for l in univariate_labels]
plt.scatter(range(len(univariate_data)), univariate_data, c=colors, alpha=0.5, s=20)
plt.title('True Labels (Red=Anomaly)'); plt.xlabel('Index'); plt.ylabel('Value')
plt.subplot(1, 2, 2)
colors = ['red' if l == 1 else 'blue' for l in y_pred_uni]
plt.scatter(range(len(univariate_data)), univariate_data, c=colors, alpha=0.5, s=20)
plt.title('Detected Anomalies (IForest)'); plt.xlabel('Index'); plt.ylabel('Value')
plt.tight_layout(); plt.show()

## 3. Multivariate Anomaly Detection

In [ ]:
X_normal = np.random.multivariate_normal([0, 0, 0], [[1, 0.5, 0.3], [0.5, 1, 0.4], [0.3, 0.4, 1]], 900)
X_anomaly = np.vstack([np.random.uniform(3, 6, (50, 3)), np.column_stack([np.random.normal(0, 0.5, 50), np.random.normal(0, 0.5, 50), np.random.normal(5, 0.5, 50)])])
X_multi = np.vstack([X_normal, X_anomaly])
y_multi = np.concatenate([np.zeros(900), np.ones(100)])
shuffle_idx = np.random.permutation(len(X_multi))
X_multi, y_multi = X_multi[shuffle_idx], y_multi[shuffle_idx]

scaler = StandardScaler()
X_multi_scaled = scaler.fit_transform(X_multi)

In [ ]:
models = {'KNN': KNN(contamination=0.1), 'LOF': LOF(contamination=0.1), 'IForest': IForest(contamination=0.1, random_state=42),
          'OCSVM': OCSVM(contamination=0.1), 'HBOS': HBOS(contamination=0.1), 'COPOD': COPOD(contamination=0.1), 'ECOD': ECOD(contamination=0.1)}
results = {}
print("Model Performance:")
for name, model in models.items():
    model.fit(X_multi_scaled)
    results[name] = {'pred': model.labels_, 'scores': model.decision_scores_, 'roc_auc': roc_auc_score(y_multi, model.decision_scores_)}
    print(f"  {name}: ROC-AUC = {results[name]['roc_auc']:.4f}")

In [ ]:
plt.figure(figsize=(12, 8))
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_multi, result['scores'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={result['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves Comparison')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Time Series Anomaly Detection

In [ ]:
t = np.linspace(0, 100, 1000)
signal = 0.05 * t + 10 * np.sin(2 * np.pi * t / 20) + np.random.normal(0, 1, 1000)
anomaly_idx = np.random.choice(1000, 50, replace=False)
signal[anomaly_idx[:25]] += np.random.choice([-1, 1], 25) * np.random.uniform(15, 25, 25)
y_ts = np.zeros(1000); y_ts[anomaly_idx] = 1

def create_features(sig, w=10):
    feats = []
    for i in range(w, len(sig)):
        win = sig[i-w:i]
        feats.append([sig[i], sig[i]-np.mean(win), (sig[i]-np.mean(win))/(np.std(win)+1e-6), np.abs(sig[i]-win[-1]), np.std(win)])
    return np.array(feats)

X_ts = create_features(signal)
y_ts_adj = y_ts[10:]
iforest_ts = IForest(contamination=0.05, random_state=42)
iforest_ts.fit(StandardScaler().fit_transform(X_ts))
print(f"Time Series ROC-AUC: {roc_auc_score(y_ts_adj, iforest_ts.decision_scores_):.4f}")

In [ ]:
plt.figure(figsize=(16, 8))
plt.subplot(2, 1, 1)
plt.plot(t, signal, 'b-', alpha=0.7)
plt.scatter(t[y_ts == 1], signal[y_ts == 1], c='red', s=100, marker='x', label='True Anomalies')
plt.title('Time Series with True Anomalies'); plt.legend()
plt.subplot(2, 1, 2)
plt.plot(t[10:], signal[10:], 'b-', alpha=0.7)
plt.scatter(t[10:][iforest_ts.labels_ == 1], signal[10:][iforest_ts.labels_ == 1], c='orange', s=100, marker='o', label='Detected')
plt.title('Detected Anomalies (IForest)'); plt.legend()
plt.tight_layout(); plt.show()

## 5. Credit Card Fraud Detection Use Case

In [ ]:
n_normal, n_fraud = 4900, 100
normal_tx = pd.DataFrame({'amount': np.random.lognormal(4, 1, n_normal), 'time_gap': np.random.exponential(24, n_normal),
                          'distance': np.random.exponential(10, n_normal), 'is_fraud': 0})
fraud_tx = pd.DataFrame({'amount': np.random.lognormal(6, 1.5, n_fraud), 'time_gap': np.random.exponential(2, n_fraud),
                         'distance': np.random.exponential(100, n_fraud), 'is_fraud': 1})
df = pd.concat([normal_tx, fraud_tx]).sample(frac=1, random_state=42).reset_index(drop=True)
X_fraud = StandardScaler().fit_transform(df[['amount', 'time_gap', 'distance']])
y_fraud = df['is_fraud'].values
print(f"Dataset: {len(df)} transactions, {y_fraud.sum()} frauds ({100*y_fraud.mean():.2f}%)")

In [ ]:
fraud_models = {'IForest': IForest(contamination=0.02, random_state=42), 'LOF': LOF(contamination=0.02), 'COPOD': COPOD(contamination=0.02)}
print("\nFraud Detection Results:")
for name, model in fraud_models.items():
    model.fit(X_fraud)
    roc = roc_auc_score(y_fraud, model.decision_scores_)
    tp = np.sum((model.labels_ == 1) & (y_fraud == 1))
    print(f"  {name}: ROC-AUC={roc:.4f}, Detected={tp}/{y_fraud.sum()} frauds")

In [ ]:
best_model = IForest(contamination=0.02, random_state=42)
best_model.fit(X_fraud)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_fraud, best_model.labels_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
axes[0].set_title('Confusion Matrix (IForest)')
fpr, tpr, _ = roc_curve(y_fraud, best_model.decision_scores_)
axes[1].plot(fpr, tpr, 'b-', label=f'IForest (AUC={roc_auc_score(y_fraud, best_model.decision_scores_):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Model Comparison Summary

In [ ]:
comparison = pd.DataFrame([{'Model': n, 'ROC-AUC': r['roc_auc'], 'Avg Precision': average_precision_score(y_multi, r['scores'])} for n, r in results.items()])
print("\nModel Comparison (Multivariate Data):")
print(comparison.sort_values('ROC-AUC', ascending=False).to_string(index=False))

## 7. Conclusion

### Key Findings:
- **IForest** and **COPOD** generally perform well across different scenarios
- **LOF** is effective for local anomalies
- **ECOD** is fast and parameter-free

### Use Cases Demonstrated:
- Univariate anomaly detection
- Multivariate anomaly detection  
- Time series anomaly detection
- Credit card fraud detection

### Best Practices:
- Standardize features before detection
- Set contamination rate based on domain knowledge
- Use ROC-AUC and Average Precision for evaluation
- Compare multiple models

In [ ]:
print("="*60)
print("ANOMALY DETECTION WITH PyOD - COMPLETE")
print("="*60)
print("\n✓ Univariate anomaly detection")
print("✓ Multivariate anomaly detection")
print("✓ Time series anomaly detection")
print("✓ Credit card fraud use case")
print("✓ Multiple model comparison")
print("✓ Comprehensive evaluation metrics")